# Introducción

En este cuaderno estaremos extendiendo la funcionalidad de extracción de las coordenadas GPS y el nombre de un fichero que contiene numerosos talleres con el objetivo de geolocalizarlos en un mapa.

Para ello primero importaremos e instalaremos las librerías y dependencias necesarias

In [0]:
import csv
import pandas as pd
from geopy.geocoders import ArcGIS
import folium

Primeramente cargamos los datos del fichero csv que hemos arrastrado a la carpeta de trabajo

In [0]:
datos = []

csvfile = open("garage_locations.csv", encoding="utf8")
reader = csv.reader(csvfile)

for row in reader:
    datos.append(row)

datos

Ahora procederemos a extraer la información del nombre del taller y sus coordenadas GPS de cada uno de los registros del fichero

In [0]:
listacompleta = []

for x in range(0, len(datos)):

  cadena = str(datos[x])

  subcadena1 = r"LatLng("
  subcadena2 = r"),icon:"
  subcadena3 = r"<h1 class="
  subcadena4 = r"\' + \'</h1>\' + \'"

  ilatlong = cadena.find(subcadena1)+7
  flatlong = cadena.find(subcadena2)
  cadena_1 = cadena[ilatlong:flatlong]
  latlong = cadena_1.split(sep=",")

  inombre = cadena.find(subcadena3)+26
  fnombre = cadena.find(subcadena4)
  nombre = cadena[inombre:fnombre]

  fila = [nombre, (latlong[0]), (latlong[1])]

  listacompleta.append(fila)

listacompleta

Ahora procederemos a guardar el listado de talleres en un DataFrame con tres features:

* Nombre del Taller
* Latitud
* Longitud

In [0]:
df = pd.DataFrame()
df = pd.DataFrame(listacompleta, columns=['Nombre del Taller', 'Latitud', 'Longitud'])
df['Nombre del Taller'] = df['Nombre del Taller'].replace({'\\\\':''}, regex=True)
df

Procederemos a limpiar el DataFrame de valores perdidos y reseteamos los índices ya que después vamos a iterar sobre el dataframe.

In [0]:
df1 = pd.DataFrame()
df1 = df.dropna()
df1.reset_index(drop=True, inplace=True)

print(df1.info())

Ahora utilizaremos un geolocalizador para enrriquecer el DataFrame con nuevas features calculadas a partir de las coordenadas GPS.

In [0]:
localización = []

geolocator = ArcGIS(user_agent="Mi_Geolocalizador")

for index, row in df1.iterrows():
  location = geolocator.reverse((row['Latitud'], row['Longitud']), timeout=None)
  localización.append(location.raw)

Antes de añadir las nuevas features al DataFrame comprobamos que la extracción de información haya sido correcta.

In [0]:
print(localización[0])

Añadiremos las features seleccionadas al DataFrame. Note que lo estamos haciendo sobre un nuevo DataFrame para dejar una copia intacta del DataFrame original.

In [0]:
Calle = []
Codigo_Postal = []
Ciudad = []
Departamento = []

for i in range(0, len(df1)):
  road=[localización[i]['Address']]
  postcode=[localización[i]['Postal']]
  city=[localización[i]['City']]
  department=[localización[i]['Subregion']]
  Calle.append(road)
  Codigo_Postal.append(postcode)
  Ciudad.append(city)
  Departamento.append(department)

df1['Calle'] = pd.DataFrame(Calle)
df1['Código Postal'] = pd.DataFrame(Codigo_Postal)
df1['Ciudad'] = pd.DataFrame(Ciudad)
df1['Departamento'] = pd.DataFrame(Departamento)
df1['Dirección'] = df1['Calle']+", "+df1['Código Postal']+" "+df1['Ciudad']

df1

Por último, procedemos a geolocalizar cada uno de los talleres en un mapa, añadiendo un popup con la dirección del mismo.

In [0]:
map1 = folium.Map(location=[47.6163939, 2.4402323], zoom_start=6)

tooltip = "Haz clic para ver el nombre del taller y la dirección"

df1.apply(lambda row:folium.Marker(location=[row["Latitud"], row["Longitud"]], popup=(row["Nombre del Taller"] + " " + row['Dirección']), tooltip=tooltip).add_to(map1), axis=1)
map1